<a href="https://colab.research.google.com/github/DeoliAlice/studio-elia-site/blob/main/Prob_de_Transportes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 47.4 MB/s eta 0:00:00


In [2]:
# importar pacotes
import pandas as pd
from ortools.linear_solver import pywraplp

In [3]:
class ProblemaDeTransporte:

    # constructor da classe
    def __init__(self,
                 nome_do_arquivo_excel,
                 nome_da_aba_custos_unitarios,
                 nome_da_aba_dados_das_origens,
                 nome_da_aba_demandas):
        # o que é executado quando um objeto dessa classe é criado
        self.nome_do_arquivo_excel = nome_do_arquivo_excel
        self.nome_da_aba_custos_unitarios = nome_da_aba_custos_unitarios
        self.nome_da_aba_dados_das_origens = nome_da_aba_dados_das_origens
        self.nome_da_aba_demandas = nome_da_aba_demandas
        self.ler_planilha()

    def ler_planilha(self):
        # Importa dados da planilha para DataFrames Python
        self.df_custos_unitarios = pd.read_excel(self.nome_do_arquivo_excel,
                                                 self.nome_da_aba_custos_unitarios,
                                                 index_col=0)

        self.df_dados_das_origens = pd.read_excel(self.nome_do_arquivo_excel,
                                                  self.nome_da_aba_dados_das_origens,
                                                  index_col=0)

        self.df_demandas = pd.read_excel(self.nome_do_arquivo_excel,
                                         self.nome_da_aba_demandas,
                                         index_col=0)

        print("Custos Unitários de Envio \n",
              self.df_custos_unitarios,
              "\n\n\n")

        print("Dados das Origens \n",
              self.df_dados_das_origens,
              "\n\n\n")

        print("Demandas dos Destinos \n",
              self.df_demandas,
              "\n\n\n")

        return

    def montar_modelo_matematico(self):
        # conjuntos dos índices (i e j)
        origens = list(self.df_custos_unitarios.index.values)
        destinos = list(self.df_custos_unitarios.columns.values)

        # resolvedor
        self.solver = pywraplp.Solver.CreateSolver("SCIP")

        # variáveis de decisão (x_{ij})
        infinito = self.solver.infinity()
        self.x = {}
        for origem in origens:
            self.x[origem] = {}
            for destino in destinos:
                self.x[origem][destino] = self.solver.IntVar(0,
                                                             infinito,
                                                             "x[%s,%s]" % (origem, destino))

        # restrições
        for origem in origens:
            lado_esquerdo = self.solver.Sum([self.x[origem][destino] for destino in destinos])
            lado_direito = self.df_dados_das_origens.loc[origem, "estoque"]
            self.solver.Add(lado_esquerdo <= lado_direito)

        for destino in destinos:
            lado_esquerdo = self.solver.Sum([self.x[origem][destino] for origem in origens])
            lado_direito = self.df_demandas.loc["demandas", destino]
            self.solver.Add(lado_esquerdo >= lado_direito)

        # função objetivo
        self.solver.Minimize(self.solver.Sum([self.df_custos_unitarios.loc[origem, destino] * self.x[origem][destino] for origem in origens for destino in destinos]))
        return

    def resolver_modelo_matematico(self):
        print(f"Resolvendo com {self.solver.SolverVersion()}")
        self.status = self.solver.Solve()

        if self.status == pywraplp.Solver.OPTIMAL:
            print(f"Prob. resolvido em {self.solver.wall_time():d} milissegundos")
            print(f"Prob. resolvido em {self.solver.iterations():d} iterações")
            print(f"Prob. resolvido em {self.solver.nodes():d} nós branch-and-bound")
        else:
            print("O problema não tem uma solução ótima.")

        return

    def mostrar_resposta(self):
        # Imprime modelo matemático
        # Imprime Função Objetivo
        print('\nFUNCÃO OBJETIVO:')
        print("Min z = ", end="")
        for origem in self.df_custos_unitarios.index:
            for destino in self.df_custos_unitarios.columns:
                custo = self.df_custos_unitarios.loc[origem, destino]
                print(f" + {custo:.2f}·x[{origem},{destino}]")

        # Imprime as restricções
        print('\nRESTRIÇÕES:')
        print('Restrições de oferta:')
        for origem in self.df_dados_das_origens.index.values:
            lado_esquerdo = self.solver.Sum([self.x[origem][destino] for destino in self.df_custos_unitarios.columns.values])
            lado_direito = self.df_dados_das_origens.loc[origem, "estoque"]
            print(f"{lado_esquerdo} <= {lado_direito}")

        print('Restrições de procura:')
        for destino in self.df_demandas.columns.values:
            lado_esquerdo = self.solver.Sum([self.x[origem][destino] for origem in self.df_custos_unitarios.index.values])
            lado_direito = self.df_demandas.loc["demandas", destino]
            print(f"{lado_esquerdo} >= {lado_direito}")

        # Imprime as variáveis
        print('\nVARIAVEIS:')
        for origem in self.df_custos_unitarios.index.values:
            for destino in self.df_custos_unitarios.columns.values:
                print(f"{self.x[origem][destino]}")

        if self.status == pywraplp.Solver.OPTIMAL:
            print('\n- - - - - - - - - - - - - - - - - - - - -')
            print('\nSOLUCÃO:')
            print("OBJETIVO =", self.solver.Objective().Value())

            print('\nRESTRIÇÕES:')
            for origem in self.df_custos_unitarios.index.values:
                soma = 0
                for destino in self.df_custos_unitarios.columns.values:
                    soma += self.x[origem][destino].solution_value()
                print(f"Total de envio de {origem} = {soma}")

            for destino in self.df_custos_unitarios.columns.values:
                soma = 0
                for origem in self.df_custos_unitarios.index.values:
                    soma += self.x[origem][destino].solution_value()
                print(f"Total recebido por {destino} = {soma}")

            # Imprime os valores das variáveis
            print('\nVARIÁVEIS:')
            for origem in self.df_custos_unitarios.index.values:
                for destino in self.df_custos_unitarios.columns.values:
                    print(f"{self.x[origem][destino]} = {self.x[origem][destino].solution_value()}")
        else:
            print("O problema não tem uma solução ótima.")
        return

In [4]:
problema_cidades_rn = ProblemaDeTransporte(
    'Problema de Transporte e de Localização de Facilidades - Cidades do RN - 8 para 12.xlsx',
    'custos unitários de envio',
    'dados das origens',
    'demandas')

problema_cidades_rn.montar_modelo_matematico()
problema_cidades_rn.resolver_modelo_matematico()
problema_cidades_rn.mostrar_resposta()

Custos Unitários de Envio 
                          Tenente Ananias  Carnaubais  Umarizal  \
Natal                           2.085290    1.071215  1.746070   
Mossoró                         1.037010    0.371855  0.594010   
Parnamirim                      2.029965    1.138790  1.690745   
São Gonçalo do Amarante         2.017255    1.003180  1.678035   
Macaíba                         1.938230    1.057790  1.599010   
Ceará-Mirim                     1.921355    0.907280  1.582135   
Caicó                           0.724265    0.808105  0.572945   
Açu                             0.999985    0.147915  0.660765   

                         Espírito Santo   Grossos  Serra Caiada  Bom Jesus  \
Natal                          0.374310  1.686325      0.468415   0.330745   
Mossoró                        1.487900  0.225460      1.218485   1.237495   
Parnamirim                     0.281825  1.631000      0.375935   0.287975   
São Gonçalo do Amarante        0.392575  1.618290      0.347940  